# pathoqc-veri: iki veri setini Kaggle'a al

Bu not defteri **bir kez** çalışır. İki veri setinin zip dosyasını yayıncı ekibin resmî Google Drive
bağlantılarından (https://github.com/ChuHan89/WSSS-Tissue) indirir ve `/kaggle/working` klasörüne koyar.

`/kaggle/working` klasörüne yazılan dosyalar, not defteri kaydedilince **çıktı** olarak saklanır.
Eğitim not defterleri bu çıktıyı girdi olarak bağlar; böylece her eğitimde yeniden indirme yapılmaz.

GPU kullanmaz. İnternet açık olmalıdır.

## 1. İndir

`gdown`, Google Drive'daki herkese açık dosyaları kimlik numarasıyla indirir.

In [ ]:
!pip -q install -U gdown
!gdown 1WyWfv-JHjLwkHOiN8blINhBOkKi--iA1 -O /kaggle/working/LUAD-HistoSeg.zip
!gdown 1sCK8Cqsf4yCHzzIx98vitocES2L_tkjn -O /kaggle/working/BCSS-WSSS.zip
!ls -l /kaggle/working

## 2. Dosyalar tam indi mi?

Boyutu, 19 Kasım 2026'da Colab'da indirilen dosyaların boyutuyla karşılaştırır.
SHA-256 özeti dosyanın parmak izidir; ileride dosya değişirse bu satırdan anlaşılır.

In [ ]:
import hashlib
from pathlib import Path

BEKLENEN_BOYUT = {
    "LUAD-HistoSeg.zip": 2101270056,
    "BCSS-WSSS.zip": 3097415608,
}

for ad, beklenen in BEKLENEN_BOYUT.items():
    dosya = Path("/kaggle/working") / ad
    boyut = dosya.stat().st_size
    ozet = hashlib.sha256()
    with open(dosya, "rb") as f:
        for parca in iter(lambda: f.read(1 << 20), b""):
            ozet.update(parca)
    print(ad, "| boyut:", boyut, "| sha256:", ozet.hexdigest())
    assert boyut == beklenen, f"{ad}: boyut {boyut}, beklenen {beklenen}. İndirme eksik ya da dosya değişmiş."

## 3. Aç ve say

Zip'leri geçici klasöre (`/kaggle/tmp`) açar. Bu klasör çıktıya dahil edilmez; sadece saymak içindir.
Sayılar makalelerdeki değerlerle aynı değilse not defteri hata verip durur.

In [ ]:
import zipfile
from collections import Counter
from pathlib import Path

TMP = Path("/kaggle/tmp/veri")
TMP.mkdir(parents=True, exist_ok=True)
for ad in BEKLENEN_BOYUT:
    with zipfile.ZipFile(Path("/kaggle/working") / ad) as z:
        z.extractall(TMP)

RESIM = {".png", ".jpg", ".jpeg"}


def gercek_resim(dosya):
    # "._ad.png" dosyaları ve "__MACOSX" klasörü zip Mac'te hazırlandığı için gelir; resim değildir.
    return (
        dosya.is_file()
        and dosya.suffix.lower() in RESIM
        and not dosya.name.startswith("._")
        and "__MACOSX" not in dosya.parts
    )


sayim = Counter()
for dosya in TMP.rglob("*"):
    if gercek_resim(dosya):
        sayim[str(dosya.parent.relative_to(TMP))] += 1

for klasor in sorted(sayim):
    print(f"{sayim[klasor]:>7}  {klasor}")

BEKLENEN_SAYI = {
    "LUAD-HistoSeg/training": 16678,
    "LUAD-HistoSeg/val/img": 300,
    "LUAD-HistoSeg/val/mask": 300,
    "LUAD-HistoSeg/test/img": 307,
    "LUAD-HistoSeg/test/mask": 307,
    "BCSS-WSSS/training": 23422,
    "BCSS-WSSS/val/img": 3418,
    "BCSS-WSSS/val/mask": 3418,
    "BCSS-WSSS/test/img": 4986,
    "BCSS-WSSS/test/mask": 4986,
}
for klasor, beklenen in BEKLENEN_SAYI.items():
    assert sayim[klasor] == beklenen, f"{klasor}: {sayim[klasor]} dosya var, beklenen {beklenen}"
print("Tüm sayılar doğru.")